# Energy Reconstruction Validation -- calibrated reco energy vs true energy

Applies the charge-to-energy calibrations fitted by `EnergyFitting.ipynb` to the
selected reco clusters and asks the downstream question: does the calibrated
**reco energy spectrum** look like the **true energy spectrum** of the true
neutrino clusters?

Same structure and knobs as `AnalysisDistributions/Reco_Distributions.ipynb`
(files/events selectors, per-level draw switches, same cuts), with the
**beam-window cut ON** -- the in-spill, neutrino-dominated reco population.

**Plots**, at event, file and job level:

| directory | contents |
|---|---|
| `reco/{linear,quadratic,saturating}_fit/` | calibrated reco energy of all selected reco clusters, 500 MeV bins -- one directory per calibration |
| `true/all_true_neutrinos/` | true energy of all true neutrino clusters (in and out of volume), 500 MeV bins |
| `reco_true_comparison/{linear,quadratic,saturating}_fit/` | the two overlaid in the same bins, one directory per calibration |

Calibration parameters are **not re-fitted here** -- they are set in the
configuration cell, with a note recording which `EnergyFitting.ipynb` run
produced them. `calibrations_used.txt` is written beside the plots so a
directory of spectra always states what separates them.

**Two things to keep in mind when reading the comparison.** The two populations
are not the same objects: the reco side is *every* selected cluster, cosmic and
neutrino alike (there is no truth on the reco side to separate them), while the
true side is neutrino clusters only. And the calibrations were fitted on the
*before*-beam-window-cut sample and are being applied to the *after*-cut one --
that transfer is the point of a validation, but a mismatch can come from it
rather than from the model's shape. Read the comparison for the shape of the
bulk, not for a bin-by-bin match.

Drawing is delegated to `AnalysisDistributions/draw_variables.py`, the
same module behind the `Reco_Distributions.ipynb` plots, so these come out in
identical style rather than as a second implementation that drifts from it.


In [20]:
# Run scope -- same knobs as Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb.
# Charge-light matching is a combined-APA evaluation (img-global / sed-sce are
# already global across APAs) -- no per-APA/face looping.
files     = "all"   # "all", or 1/2/3/... to limit the number of file subdirectories processed
events    = "all"   # "all", or 1/2/3/... to limit the number of events processed per file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
# Set to None to process all files/events, or specify to run only specific ones
# Example: target_file = "file0", target_event = 3  (to process only file0, event 3)
target_file  = None   # Set to "file0", "file1", etc. to process specific file only
target_event = None   # Set to a SINGLE event number (0, 1, ..., 9); use target_event_range below for a span

# Range of event numbers to run, inclusive on both ends: (1, 5) runs events
# 1,2,3,4,5. None runs every event. Applied on top of target_event, so leave
# target_event = None when using a range.
target_event_range = None   # e.g. (1, 5) for events 1..5

# Range of file INDICES to run, inclusive on both ends: (6, 9) runs file6, file7,
# file8, file9. None runs every file. Matched on the number at the end of the
# directory name, NOT on position in the list -- the directories sort
# lexicographically (file0, file1, file10, file11, file2, ...), so a positional
# slice would pick the wrong files. The `files = N` knob above still takes the
# first N in lexicographic order.
# NOTE: target_file (above) is applied too, so set it to None when using a range,
# otherwise only the one file that satisfies both runs.
target_file_range = None   # e.g. (6, 9) for file6..file9

# Fail fast rather than silently processing nothing: `evt != target_event` can
# never be False for a tuple, so target_event = (1, 5) would skip every event.
if isinstance(target_event, (tuple, list)):
    raise ValueError(
        f"target_event={target_event} is a range, but target_event takes a single event number. "
        f"Use target_event_range={tuple(target_event)} and target_event = None instead.")


# ========================================================================
# Decide which levels to draw
# ========================================================================
# Every level draws the SAME plots; they differ only in how many clusters are
# pooled into each. An event-level energy spectrum holds a handful of clusters --
# useful for checking one event, noise for physics. Turn it off for a
# full-statistics run.
b_draw_event_level_plots = False   # one plot per event
b_draw_file_level_plots  = False   # one plot per file
b_draw_job_level_plots   = True   # one set of spectra for the whole job


In [21]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os
import time
from datetime import datetime

np.set_printoptions(linewidth=1000)

# This notebook lives in EnergyReconstruction/, one level below the repository
# root where the pipeline modules and the input trees are. Resolve both
# explicitly so the notebook runs whether Jupyter was started in this directory
# (the usual case) or at the repository root.
NB_DIR = Path.cwd()
if NB_DIR.name != "EnergyReconstruction":
    NB_DIR = NB_DIR / "EnergyReconstruction"
NB_DIR = NB_DIR.resolve()
REPO_ROOT = NB_DIR.parent

# AnalysisDistributions is on the path too: this notebook draws with
# that directory's draw_variables.py, so its plots are the same objects in the
# same style as Reco_Distributions.ipynb's rather than a second implementation.
DISTRIBUTIONS_DIR = REPO_ROOT / "AnalysisDistributions"

for path in (str(REPO_ROOT), str(NB_DIR), str(DISTRIBUTIONS_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Notebook directory: {NB_DIR}")
print(f"Repository root:    {REPO_ROOT}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Job started at: 2026-08-04 17:00:30
Notebook directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/EnergyReconstruction
Repository root:    /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction


In [22]:
# Pipeline modules (repository root) -- imported and used UNCHANGED. The 1-to-1
# pairing IS computed: the pair_true_reco_clusters/ version of every plot needs
# to know which reco cluster was matched to which true neutrino, and completeness
# and purity are computed only because MatchTrueToReco1to1 needs them.
from readfiles import ensure_data_extracted, read_charge_light_files_for_event, flatten_mc_tree
from selections import (
    GroupClustersByID, build_true_points_charge_light, apply_deadarea_cut_true_charge_light,
    reassign_cluster_ID_true_charge_light, reassign_cluster_ID_reco,
    apply_energy_cutoff, apply_true_pointwise_energy_cutoff,
    apply_energy_cutoff, apply_min_true_points_cutoff, apply_min_reco_points_cutoff,
    apply_wire_readout_sensitive_yz_plane_cut_true, apply_wire_readout_sensitive_yz_plane_cut_reco,
)
from cluster_category import cluster_category
from completeness_purity_estimate import EvaluateCompleteness, EvaluatePurity
from clusterpairmatching import MatchTrueToReco1to1
from metadata import (
    build_cluster_flash_metadata, build_img_cluster_flash_metadata,
    add_metadata_true_reco_pair_cluster, build_neutrino_vertex_records,
)
from DrawRecoTrueFlashes import BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US

# Per-cluster record builders from AnalysisDistributions -- the same
# ones Reco_Distributions.ipynb uses, so the clusters behind these spectra are
# the same objects described the same way.
from draw_variables import (
    build_reco_cluster_variable_records, build_true_cluster_variable_records,
    select_true_neutrino_records,
)

# The new module for this notebook (EnergyReconstruction/draw_energy_validation.py):
# applies a calibration to each reco cluster's charge and draws the spectra.
from draw_energy_validation import (
    draw_energy_validation_versions, select_matched_pair_records,
    write_calibration_summary, reco_energy_from_charge, format_model,
    ALL_MODELS, ENERGY_BIN_WIDTH_MEV, VERSION_DIRNAME,
)


In [23]:
# Configuration: Parent directory containing multiple file subdirectories (file0/, file1/, ...)
#
# Expected structure (per file subdirectory). The preprocessed tree has no zip --
# its data/ is already present, so ensure_data_extracted() below simply no-ops.
# PARENT_DIR/
#   file0/data/0/0-img-global.json                       (reco clusters, imaging level)
#   file0/data/0/0-clustering-global.json                (reco clusters, post charge-light matching)
#   file0/data/0/0-sed-smear_readout.json                (true clusters)
#   file0/data/0/0-mc.json                               (particle truth ancestry tree)
#   file0/data/0/0-op.json                               (optical/light info)
#   file0/data/1/, 2/, ... (one subdirectory per event)

# The DEAD-AREA-PREPROCESSED tree, produced once by preprocess_deadarea_cut.py.
# Its true-point files already have the dead-area cut applied, which is why
# Apply_deadarea_cut is False below. Read that script's docstring, or
# DEADAREA_PREPROCESSING.txt inside the tree, before switching this back to the
# raw tree: the two are NOT interchangeable.
PARENT_DIR = REPO_ROOT / "Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut"

# Number of files/events to process (convert the 'files'/'events' knobs above)
num_files_to_process  = None if files  == "all" else files
num_events_to_process = None if events == "all" else events

# ========================================================================
# THE CALIBRATIONS BEING VALIDATED
# ========================================================================
# Fitted by EnergyFitting.ipynb -- copied here rather than re-fitted, so this
# notebook validates a calibration instead of producing one. Update these
# together with CALIBRATION_SOURCE whenever the fit is redone.
#
# Parameters are in the order each formula takes them:
#   linear      E = a*Q                  -> (a,)
#   quadratic   E = a*Q + b*Q^2          -> (a, b)
#   saturating  E = a*Q / (1 + c*Q)      -> (a, c)
CALIBRATION_SOURCE = ("EnergyFitting.ipynb, config charge_to_5e7ADC "
                      "(charge <= 5e7 ADC, full energy range, x rebinned by 2), "
                      "fitted on EnergyReconstruction_BeforeTimeWindowCut / "
                      "combined_apa_20260804_005916_allfiles / all_true_clusters")

MODEL_PARAMS = {
    'linear':     (2.91499e-05,),
    'quadratic':  (4.66830e-05, -5.63991e-13),
    'saturating': (5.47869e-05, 3.02913e-08),
}

# Energy bin width for every spectrum here, MeV. Coarser than the 100 MeV the
# calibrations were fitted with: these spectra hold far fewer clusters than
# that histogram did, and at 100 MeV they come out one cluster per bin with no
# shape to compare.
ENERGY_BIN_WIDTH = 500.0

# ========================================================================
# SELECTION PARAMETERS -- the same values as
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb. The completeness and
# purity radii matter only through MatchTrueToReco1to1, which needs them to
# decide which reco cluster is a true neutrino's best match -- that pairing is
# what pair_true_reco_clusters/ is drawn from.
# ========================================================================
radius_completeness         = 2
radius_purity_xz          = 2
radius_purity_yz          = 5
radius_purity_xy          = 5
min_recopoints_threshold  = 5

# min_true_points_cutoff / min_reco_points_cutoff are DISABLED: this format's
# point clouds are much sparser than the old imaging-based reconstruction --
# real neutrino clusters have been seen with as few as 13 points -- so the old
# threshold (200) would delete real signal clusters outright.
#
# min_cluster_energy IS applied: sed-smear's per-point 'e' field (MeV) is a
# genuine energy deposit, so the old threshold (100 MeV) carries over directly.
min_cluster_energy        = 100     # APPLIED (Apply_energy_cutoff = True below)
min_true_point_energy     = 0.02    # MeV per POINT (Apply_trueenergy_pointwise_cutoff below)
min_true_points_cutoff    = 200     # NOT APPLIED (Apply_min_true_points_cutoff = False below)
min_reco_points_cutoff    = 200     # NOT APPLIED (Apply_min_reco_points_cutoff = False below)

Apply_energy_cutoff                         = True
Apply_trueenergy_pointwise_cutoff           = True    # drop true POINTS below min_true_point_energy
Apply_min_true_points_cutoff                = False
Apply_min_reco_points_cutoff                = False
Apply_wire_readout_sensitive_xz_plane_cut   = True
Apply_time_window_cut                       = False   # must stay disabled -- no per-point true time in this format
# BEAM-WINDOW (time) CUT -- ON here. Only reco clusters whose bridged flash time
# lies inside [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US] = [0.33, 1.93] us survive,
# i.e. the in-spill, neutrino-dominated population. That is the population whose
# calibrated energy spectrum is worth comparing against the true neutrino
# spectrum; without the cut the reco side is dominated by cosmic clusters that
# have no counterpart on the true side at all.
#
# The output directory below follows the flag, so before-cut and after-cut runs
# never land in the same tree.
Apply_beam_window_cut                       = True
# The dead-area cut is APPLIED, just not here: PARENT_DIR above is the tree
# preprocess_deadarea_cut.py already cut. Set this True only if you point
# PARENT_DIR back at a raw tree.
Apply_deadarea_cut                          = False

# Wire-readout sensitive volume (detector geometry, unit: cm). Also the bounds
# for the vertex_in_volume flag carried onto each pair record.
x_min = -250.0
x_max = 250.0
y_min = -200.0
y_max = 200.0
z_min = 0.15
z_max = 500.85

# Metadata label only -- there is no 2-view/3-view distinction in the
# charge-light format, so this is just a constant.
view = "combined"

# Output directory: inside EnergyReconstruction, so this notebook's output tree
# is self-contained and never shares a directory with the other notebooks' plots.
# The leaf name follows Apply_beam_window_cut -- the two settings describe
# different reco populations, and mixing their runs in one tree would make a
# timestamped directory the only clue as to which is which.
PLOTBASEDIR = NB_DIR / "multi_file_plots_charge_light_matching" / (
    "EnergyReconstruction_AfterTimeWindowCut" if Apply_beam_window_cut
    else "EnergyReconstruction_BeforeTimeWindowCut")
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    print(f"\n⚡ SELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        print(f"  Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        print(f"  Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")

print("\nPlots, drawn twice -- once per population:")
print(f"- {VERSION_DIRNAME['all']}/   every selected reco cluster vs every true neutrino")
print(f"- {VERSION_DIRNAME['pairs']}/             only the 1-to-1 matched true-reco pairs")
print(f"  each with reco/, true/ and reco_true_comparison/, in {ENERGY_BIN_WIDTH:.0f} MeV bins")
print("\nCalibrations:")
for model in ALL_MODELS:
    print(f"- {model:<11s} {format_model(model, MODEL_PARAMS[model])}")
print(f"  source: {CALIBRATION_SOURCE}")

print("\nCuts applied:")
if Apply_energy_cutoff:
    print(f"- Energy cutoff applied (threshold {min_cluster_energy} MeV, using sed-smear's per-point 'e' field)")
if Apply_wire_readout_sensitive_xz_plane_cut:
    print(f"- Wire readout sensitive xz plane cut applied")
if Apply_beam_window_cut:
    print(f"- Beam window cut applied to RECO clusters only ({BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us flash time)")
else:
    print(f"- Beam window cut NOT applied -- every selected reco cluster is available for matching, "
          f"in-spill or not")
if Apply_deadarea_cut:
    print(f"- Dead area cut applied HERE")
else:
    print(f"- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR)")

# ========================================================================
# ONE-TIME EXTRACTION
# ========================================================================
# ensure_data_extracted() only unzips if that file's data/ folder doesn't
# already exist, so re-running this notebook never re-extracts.
if PARENT_DIR.exists():
    for subdir in sorted(PARENT_DIR.iterdir()):
        if subdir.is_dir():
            ensure_data_extracted(subdir)
else:
    print(f"Error: Parent directory {PARENT_DIR} does not exist")


Configuration:
Parent directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
Plot base directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/EnergyReconstruction/multi_file_plots_charge_light_matching/EnergyReconstruction_AfterTimeWindowCut
Files to process: all
Events to process: all

Plots, drawn twice -- once per population:
- all_true_all_selected_reco_clusters/   every selected reco cluster vs every true neutrino
- pair_true_reco_clusters/             only the 1-to-1 matched true-reco pairs
  each with reco/, true/ and reco_true_comparison/, in 500 MeV bins

Calibrations:
- linear      E = 2.915e-05*Q
- quadratic   E = 4.668e-05*Q - 5.64e-13*Q^2
- saturating  E = 5.479e-05*Q / (1 + 3.029e-08*Q)
  source: EnergyFitting.ipynb, config charge_to_5e7ADC (charge <= 5e7 ADC, full energy range, x rebinned by 2), fitted on EnergyReconstruction_BeforeTimeWindowCut / combined_a

In [24]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories (file0/, file1/, etc.).
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs


def file_index_from_name(name):
    """
    Trailing integer of a file directory name ("file10" -> 10), or None if it
    has no trailing digits. Used by target_file_range so files are selected by
    their real index rather than by position in the lexicographically sorted
    list (file0, file1, file10, file11, file2, ...).
    """
    digits = ""
    for ch in reversed(name):
        if not ch.isdigit():
            break
        digits = ch + digits
    return int(digits) if digits else None


def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"

    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []

    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                events.append(int(item.name))
            except ValueError:
                pass

    return sorted(events)


# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        detected_events = detect_events_in_directory(input_dir)
        if detected_events:
            print(f"  {input_dir.name}/data/: {len(detected_events)} events ({min(detected_events)}-{max(detected_events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")


Scanning parent directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
------------------------------------------------------------
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file1
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file10
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file11
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file2

In [25]:
# ============================================================================
# MAIN PROCESSING LOOP -- combined-APA energy reconstruction validation
# ============================================================================
# The selection chain below is a copy of
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb's, truncated after the
# 1-to-1 pairing: true side = sed-smear_readout grouped by REAL_CLUSTER_ID and
# reassigned to 99990+nu_idx (neutrino, one cluster per interaction) / avg-X
# (cosmic); reco side = clustering-global grouped by REAL_CLUSTER_ID (NOT
# cluster_id, which can merge physically distinct tracks), beam-window cut,
# fiducial cut, then relabelled by avg-X via reassign_cluster_ID_reco.
#
# No true-reco matching happens here: every plot is a spectrum of one population
# on its own. The reco side contributes each selected cluster's total charge,
# which the calibrations turn into an energy; the true side contributes each
# true neutrino cluster's energy, summed from the sed points -- the same
# quantity every cut and completeness number in this pipeline uses.

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = PLOTBASEDIR / f"combined_apa_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n{'='*70}")
print(f"Output directory: {output_dir}")
print(f"{'='*70}\n")

job_reco_var_records   = []    # one record per selected reco cluster
job_true_var_records   = []    # one record per selected true cluster
job_pair_metadata_list = []    # per 1-to-1 true-reco pair (add_metadata_true_reco_pair_cluster)
job_vertex_records     = []    # per true neutrino interaction (build_neutrino_vertex_records)
total_events_processed  = 0
total_files_processed   = 0

for file_idx, input_dir in enumerate(input_directories):
    input_file_name = input_dir.name

    # SELECTIVE FILTERING: Skip files that don't match target_file
    if target_file is not None and input_file_name != target_file:
        print(f"Skipping {input_file_name} (target: {target_file})")
        continue

    # SELECTIVE FILTERING: Skip files outside target_file_range (inclusive both
    # ends, matched on the directory name's trailing index -- see
    # file_index_from_name). Applied on top of target_file, not instead of it.
    if target_file_range is not None:
        file_idx = file_index_from_name(input_file_name)
        range_low, range_high = target_file_range
        if file_idx is None or not (range_low <= file_idx <= range_high):
            print(f"Skipping {input_file_name} (target range: file{range_low}..file{range_high})")
            continue

    print(f"\n{'='*70}")
    print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir}")
    print(f"{'='*70}")

    file_output_dir = output_dir / input_file_name
    file_output_dir.mkdir(parents=True, exist_ok=True)

    file_reco_var_records   = []
    file_true_var_records   = []
    file_pair_metadata_list = []
    file_vertex_records     = []

    events_list = detect_events_in_directory(input_dir)
    if not events_list:
        print(f"No events found in {input_dir}, skipping...")
        continue

    event_low = min(events_list)
    event_high = max(events_list) + 1 if num_events_to_process is None else event_low + num_events_to_process

    print(f"Processing events {event_low} to {event_high-1}\n")
    total_files_processed += 1

    # Start of event loop
    for evt in range(event_low, event_high):
        # SELECTIVE FILTERING: Skip events that don't match target_event
        if target_event is not None and evt != target_event:
            continue

        # SELECTIVE FILTERING: Skip events outside target_event_range (inclusive
        # both ends). Applied on top of target_event, not instead of it.
        if target_event_range is not None:
            event_range_low, event_range_high = target_event_range
            if not (event_range_low <= evt <= event_range_high):
                continue

        result = read_charge_light_files_for_event(input_dir, evt)
        if result is None:
            print(f"  Event {evt}: could not read data, skipping")
            continue

        event_key        = f"{input_file_name}_{evt}"
        event_output_dir = file_output_dir / f"event_{evt:03d}"
        event_output_dir.mkdir(parents=True, exist_ok=True)

        x_true, y_true, z_true, id_true, q_true, real_id_true, e_true, nu_idx_true = result['true_clustering']
        x_clu,  y_clu,  z_clu,  id_clu,  q_clu,  real_id_clu                       = result['clustering']
        mc_tree = result['mc']
        op_data = result['op']

        # ------------------------------------------------------------------
        # FLASH RECORDS: op.json flashes attached to img-global clusters, then
        # bridged onto clustering-global clusters by point charge ('q'). Used
        # here only to identify which reco clusters are in the beam window.
        # ------------------------------------------------------------------
        event_flash_metadata_list = build_cluster_flash_metadata(
            op_data, input_file_name, evt, "Combined", event_key)
        event_img_cluster_flash_records = build_img_cluster_flash_metadata(
            result['reco'], result['clustering'], event_flash_metadata_list,
            input_file_name, evt, "Combined", event_key)

        clu_beam_window_ids = {float(r['clustering_cluster_id']) for r in event_img_cluster_flash_records
                               if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}

        # ------------------------------------------------------------------
        # TRUE POINTS: sed-smear_readout in the standard 7-column shape
        # (energy = per-point 'e' in MeV -- the y axis of these plots comes from
        # summing this column; q_true = 'nu_idx', 0=cosmic, 1/2/...=which
        # neutrino interaction), reassigned to 99990+nu_idx (neutrino) / avg-X
        # (cosmic), then cut.
        # ------------------------------------------------------------------
        true_points = build_true_points_charge_light(
            x_true, y_true, z_true, real_id_true, q_true, energy=e_true, nu_idx=nu_idx_true)
        true_points = reassign_cluster_ID_true_charge_light(true_points)

        # Snapshot BEFORE the cuts: build_neutrino_vertex_records uses it to say
        # what a removed neutrino actually deposited. Grouping only, no filtering.
        clusters_true_precut = GroupClustersByID(true_points)

        # POINT-wise first, so the cluster total the cluster cut tests is the
        # total of the points that survive.
        if Apply_trueenergy_pointwise_cutoff:
            true_points = apply_true_pointwise_energy_cutoff(true_points, min_true_point_energy)
        if Apply_energy_cutoff:
            true_points = apply_energy_cutoff(true_points, min_cluster_energy)
        if Apply_min_true_points_cutoff:
            true_points = apply_min_true_points_cutoff(true_points, min_true_points_cutoff)
        if Apply_wire_readout_sensitive_xz_plane_cut:
            true_points = apply_wire_readout_sensitive_yz_plane_cut_true(true_points, x_min, x_max, y_min, y_max, z_min, z_max)
        if Apply_deadarea_cut:
            true_points = apply_deadarea_cut_true_charge_light(true_points, output_dir=event_output_dir, event=evt, file_name=input_file_name)

        if len(true_points) == 0:
            print(f"  Event {evt}: no true points remain after cuts, skipping")
            continue

        clusters_true = GroupClustersByID(true_points)

        # ------------------------------------------------------------------
        # RECO POINTS: clustering-global (post charge-light matching), grouped
        # by REAL_CLUSTER_ID. Beam-window cut FIRST -- clu_beam_window_ids lives
        # in the real_cluster_id namespace, which reassign_cluster_ID_reco
        # destroys, so filtering after it would match nothing.
        # ------------------------------------------------------------------
        predicted_points = np.column_stack((x_clu, y_clu, z_clu, real_id_clu, q_clu))

        if Apply_beam_window_cut:
            n_reco_points_before_beam   = len(predicted_points)
            n_reco_clusters_before_beam = len(np.unique(predicted_points[:, 3])) if n_reco_points_before_beam else 0
            beam_ids_array   = np.fromiter(clu_beam_window_ids, dtype=float, count=len(clu_beam_window_ids))
            predicted_points = predicted_points[np.isin(predicted_points[:, 3], beam_ids_array)]
            print(f"  Event {evt}: beam-window cut kept "
                  f"{len(clu_beam_window_ids)}/{n_reco_clusters_before_beam} reco clusters, "
                  f"{len(predicted_points)}/{n_reco_points_before_beam} reco points")

        if Apply_min_reco_points_cutoff:
            predicted_points = apply_min_reco_points_cutoff(predicted_points, min_reco_points_cutoff)
        if Apply_wire_readout_sensitive_xz_plane_cut:
            predicted_points = apply_wire_readout_sensitive_yz_plane_cut_reco(predicted_points, x_min, x_max, y_min, y_max, z_min, z_max)

        # An event can legitimately end up with NO in-spill reco cluster. It is
        # kept rather than skipped, but reassign_cluster_ID_reco cannot take an
        # empty array, so short-circuit to an empty dict -- that event simply
        # contributes no pair.
        if len(predicted_points) == 0:
            clusters_reco = {}
            print(f"  Event {evt}: no reco cluster survives the beam-window cut")
        else:
            predicted_points = reassign_cluster_ID_reco(predicted_points)
            clusters_reco    = GroupClustersByID(predicted_points)

        # ------------------------------------------------------------------
        # 1-TO-1 TRUE-RECO PAIRING (existing functions, unchanged). Needed for
        # the pair_true_reco_clusters/ version of every plot: completeness and
        # purity are computed only because MatchTrueToReco1to1 needs them, and
        # nothing here plots either.
        # ------------------------------------------------------------------
        cluster_category_results = cluster_category(clusters_true, output_dir=None, event=evt, apa="Combined", file_name=input_file_name)
        completeness_results       = EvaluateCompleteness(clusters_true, clusters_reco, event_key, radius_completeness, min_recopoints_threshold)
        purity_results           = EvaluatePurity(clusters_true, clusters_reco, event_key, radius_purity_xz, radius_purity_yz, radius_purity_xy)

        event_matched_pairs      = MatchTrueToReco1to1(completeness_results, purity_results)
        event_pair_metadata_list = add_metadata_true_reco_pair_cluster(
            event_matched_pairs, cluster_category_results,
            file_name=input_file_name, event=evt, apa="Combined", view=view, event_key=event_key)

        # ------------------------------------------------------------------
        # TRUE NEUTRINO INTERACTION VERTICES (mc.json), joined to their true
        # cluster by nu_idx (cluster_id = 99990+nu_idx, an exact key). Not used
        # to select anything here -- every true neutrino cluster is plotted, in
        # volume or not -- but it puts the vertex_in_volume flag into the true
        # side's text table.
        # ------------------------------------------------------------------
        event_vertex_records = build_neutrino_vertex_records(
            flatten_mc_tree(mc_tree), clusters_true, input_file_name, evt, event_key,
            x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max,
            clusters_true_precut=clusters_true_precut, min_cluster_energy=min_cluster_energy)

        # ------------------------------------------------------------------
        # PER-CLUSTER RECORDS + DRAWING. The calibrations are applied inside
        # draw_all_energy_validation_plots, one copy of the reco records per
        # model, so the three spectra come from one set of clusters.
        # No flash-time lookup is passed: flash time is not plotted here, and
        # the beam-window cut that used it has already been applied above.
        # ------------------------------------------------------------------
        event_reco_var_records = build_reco_cluster_variable_records(
            clusters_reco, input_file_name, evt, event_key, "Combined")
        event_true_var_records = build_true_cluster_variable_records(
            clusters_true, input_file_name, evt, event_key, "Combined",
            vertex_records=event_vertex_records)

        if b_draw_event_level_plots:
            draw_energy_validation_versions(
                event_reco_var_records, event_true_var_records, event_pair_metadata_list,
                event_output_dir, f"Event {evt}", f"event_{evt}", "Combined",
                file_name=input_file_name, model_params=MODEL_PARAMS,
                bin_width=ENERGY_BIN_WIDTH)
            plt.close('all')

        # ------------------------------------------------------------------
        # AGGREGATE TO FILE AND JOB LEVEL
        # ------------------------------------------------------------------
        file_reco_var_records.extend(event_reco_var_records)
        file_true_var_records.extend(event_true_var_records)
        file_pair_metadata_list.extend(event_pair_metadata_list)
        file_vertex_records.extend(event_vertex_records)

        job_reco_var_records.extend(event_reco_var_records)
        job_true_var_records.extend(event_true_var_records)
        job_pair_metadata_list.extend(event_pair_metadata_list)
        job_vertex_records.extend(event_vertex_records)

        n_true_neutrino = len(select_true_neutrino_records(event_true_var_records))
        print(
            f"  Event {evt}: "
            f"selected reco clusters={len(event_reco_var_records)}, "
            f"true clusters={len(event_true_var_records)} (neutrino={n_true_neutrino}), "
            f"1-to-1 neutrino pairs={len(select_matched_pair_records(event_reco_var_records, event_true_var_records, event_pair_metadata_list)[0])}, "
            f"neutrino interactions in mc={len(event_vertex_records)}"
        )
        total_events_processed += 1

    # ========================================================================
    # FILE-LEVEL PLOTS: the same plots over every event in this file
    # ========================================================================
    if b_draw_file_level_plots:
        print(f"\n  FILE-LEVEL AGGREGATION ({input_file_name}): "
              f"{len(file_reco_var_records)} reco clusters, "
              f"{len(file_true_var_records)} true clusters "
              f"({len(select_true_neutrino_records(file_true_var_records))} neutrino)")
        draw_energy_validation_versions(
            file_reco_var_records, file_true_var_records, file_pair_metadata_list,
            file_output_dir / "file_summary", "File Level", "file", "Combined",
            file_name=input_file_name, model_params=MODEL_PARAMS,
            bin_width=ENERGY_BIN_WIDTH)
        plt.close('all')

# ============================================================================
# JOB-LEVEL PLOTS: the same plots over every file and event
# ============================================================================
print(f"\n{'='*70}")
print(f"JOB SUMMARY: {total_files_processed} file(s), {total_events_processed} event(s) processed")
print(f"Total selected reco clusters: {len(job_reco_var_records)}")
print(f"Total selected true clusters: {len(job_true_var_records)} "
      f"({len(select_true_neutrino_records(job_true_var_records))} neutrino)")
job_paired_reco, job_paired_true = select_matched_pair_records(
    job_reco_var_records, job_true_var_records, job_pair_metadata_list)
print(f"Total 1-to-1 neutrino pairs: {len(job_paired_reco)} reco / {len(job_paired_true)} true")
print(f"{'='*70}")

job_output_dir = output_dir / "job_summary"
job_output_dir.mkdir(parents=True, exist_ok=True)

if b_draw_job_level_plots:
    draw_energy_validation_versions(
        job_reco_var_records, job_true_var_records, job_pair_metadata_list,
        job_output_dir, "Job Level", "job", "Combined", model_params=MODEL_PARAMS,
        bin_width=ENERGY_BIN_WIDTH)
    plt.close('all')

# Which calibration each spectrum was drawn with, written beside the plots so a
# directory of spectra always states what separates them.
calibration_path = write_calibration_summary(MODEL_PARAMS, job_output_dir,
                                             source_note=CALIBRATION_SOURCE)
print(f"Calibrations used written to: {calibration_path}")

# ============================================================================
# JOB SUMMARY TEXT FILE -- configuration, how many pairs each stage kept, and
# the summary numbers of the plotted distribution.
# ============================================================================
from datetime import timedelta

job_finish_dt = datetime.now()
job_runtime   = time.time() - job_start_time

job_true_neutrinos = select_true_neutrino_records(job_true_var_records)

summary_lines = []
summary_lines.append("=" * 80)
summary_lines.append("JOB SUMMARY -- ENERGY RECONSTRUCTION VALIDATION")
summary_lines.append("=" * 80)
summary_lines.append(f"Generated: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")
summary_lines.append("Configuration:")
summary_lines.append(f"Parent directory: {PARENT_DIR}")
summary_lines.append(f"Plot base directory: {PLOTBASEDIR}")
summary_lines.append(f"Files to process: {files}")
summary_lines.append(f"Events to process: {events}")
if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    summary_lines.append(f"Target file: {target_file if target_file else 'all'}")
    summary_lines.append(f"Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        summary_lines.append(f"Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        summary_lines.append(f"Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")
summary_lines.append("")
summary_lines.append("Cuts (identical to Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb):")
summary_lines.append(f"  energy cutoff:            {Apply_energy_cutoff} ({min_cluster_energy} MeV)")
summary_lines.append(f"  min true points cutoff:   {Apply_min_true_points_cutoff} ({min_true_points_cutoff})")
summary_lines.append(f"  min reco points cutoff:   {Apply_min_reco_points_cutoff} ({min_reco_points_cutoff})")
summary_lines.append(f"  wire readout volume cut:  {Apply_wire_readout_sensitive_xz_plane_cut}")
summary_lines.append(f"  beam window cut (reco):   {Apply_beam_window_cut} ({BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us)")
summary_lines.append(f"  dead area cut here:       {Apply_deadarea_cut} (applied upstream when False)")
summary_lines.append(f"  volume bounds: x [{x_min}, {x_max}], y [{y_min}, {y_max}], z [{z_min}, {z_max}] cm")
summary_lines.append("")
summary_lines.append("Plots:")
summary_lines.append(f"  Drawn twice, once per population:")
summary_lines.append(f"    {VERSION_DIRNAME['all']}/   every selected reco cluster vs every true neutrino")
summary_lines.append(f"    {VERSION_DIRNAME['pairs']}/             only the 1-to-1 matched true-reco pairs")
summary_lines.append(f"  Each holding:")
summary_lines.append(f"    reco/<model>_fit/                 calibrated energy, {ENERGY_BIN_WIDTH:.0f} MeV bins")
summary_lines.append(f"    reco/reco_energy_all_models_*     all three calibrations on one axes")
summary_lines.append(f"    true/all_true_neutrinos/          true energy of the true neutrino clusters")
summary_lines.append(f"    reco_true_comparison/<model>_fit/ the two overlaid in the same bins")
summary_lines.append(f"    reco_true_comparison/*_all_models_*  all three side by side")
summary_lines.append("")
summary_lines.append("Calibrations applied to reco cluster charge:")
for model in ALL_MODELS:
    summary_lines.append(f"  {model:<12s} {format_model(model, MODEL_PARAMS[model])}")
summary_lines.append(f"  source: {CALIBRATION_SOURCE}")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("JOB-LEVEL AGGREGATION")
summary_lines.append("=" * 80)
summary_lines.append(f"Total files processed: {total_files_processed}")
summary_lines.append(f"Total events processed: {total_events_processed}")
summary_lines.append(f"Total selected reco clusters (PLOTTED in reco/): {len(job_reco_var_records)}")
summary_lines.append(f"Total selected true clusters: {len(job_true_var_records)}")
summary_lines.append(f"  of which neutrino (PLOTTED in true/): {len(job_true_neutrinos)}")
summary_lines.append(f"1-to-1 neutrino pairs (PLOTTED in {VERSION_DIRNAME['pairs']}/): "
                     f"{len(job_paired_reco)} reco / {len(job_paired_true)} true")
summary_lines.append(f"Total true neutrino interactions (mc.json): {len(job_vertex_records)}")
summary_lines.append("")

# Summary numbers of the spectra themselves: what each calibration makes of the
# reco clusters, beside the true neutrino spectrum they are meant to resemble.
if job_reco_var_records:
    charges = np.array([r['total_charge'] for r in job_reco_var_records], dtype=float)
    summary_lines.append("Selected reco clusters (job level):")
    summary_lines.append(f"  charge [ADC]: mean {charges.mean():.4g}, "
                         f"min {charges.min():.4g}, max {charges.max():.4g}")
    for model in ALL_MODELS:
        energies = reco_energy_from_charge(charges, model, MODEL_PARAMS[model])
        summary_lines.append(f"  {model:<12s} reco energy [MeV]: mean {energies.mean():.1f}, "
                             f"min {energies.min():.1f}, max {energies.max():.1f}")
    summary_lines.append("")

if job_true_neutrinos:
    true_energies = np.array([r['total_energy'] for r in job_true_neutrinos], dtype=float)
    summary_lines.append("True neutrino clusters (job level):")
    summary_lines.append(f"  true energy [MeV]: mean {true_energies.mean():.1f}, "
                         f"min {true_energies.min():.1f}, max {true_energies.max():.1f}")
    summary_lines.append("")
    summary_lines.append("The two populations are NOT the same objects -- reco is every selected")
    summary_lines.append("cluster (cosmic and neutrino alike, no truth on that side to separate")
    summary_lines.append("them), true is neutrino clusters only -- so compare the SHAPE of the bulk,")
    summary_lines.append("not the counts.")
    summary_lines.append("")

summary_lines.append("=" * 80)
summary_lines.append("JOB RUNTIME")
summary_lines.append("=" * 80)
summary_lines.append(f"Job started at:  {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Total job runtime: {timedelta(seconds=int(job_runtime))} ({job_runtime:.1f} seconds)")
summary_lines.append("=" * 80)

with open(job_output_dir / "summary.txt", "w") as f:
    f.write("\n".join(summary_lines) + "\n")

print(f"\nJob finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')} (runtime: {job_runtime:.1f}s)")
print(f"Job summary written to: {job_output_dir / 'summary.txt'}")



Output directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/EnergyReconstruction/multi_file_plots_charge_light_matching/EnergyReconstruction_AfterTimeWindowCut/combined_apa_20260804_170030


FILE 1/12: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
Processing events 0 to 9

  Event 0: beam-window cut kept 0/15 reco clusters, 0/21848 reco points
  Event 0: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 0: selected reco clusters=0, true clusters=7 (neutrino=0), 1-to-1 neutrino pairs=0, neutrino interactions in mc=1
  Event 1: beam-window cut kept 1/13 reco clusters, 740/18105 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 1: selected reco clusters=1, true clusters=6 (neutrino=1), 1-to-1 neutrino pairs=1, neutrino interactions in mc=1
  Event 2: beam-window cut kept 1/14 reco 

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(completeness_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 110: beam-window cut kept 1/17 reco clusters, 29/19670 reco points

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 110: selected reco clusters=1, true clusters=7 (neutrino=0), 1-to-1 neutrino pairs=0, neutrino interactions in mc=4


/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(completeness_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 111: beam-window cut kept 1/13 reco clusters, 3050/23839 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 111: selected reco clusters=1, true clusters=7 (neutrino=1), 1-to-1 neutrino pairs=1, neutrino interactions in mc=2
  Event 112: beam-window cut kept 0/9 reco clusters, 0/36943 reco points
  Event 112: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 112: selected reco clusters=0, true clusters=7 (neutrino=0), 1-to-1 neutrino pairs=0, neutrino interactions in mc=1
  Event 113: beam-window cut kept 0/18 reco clusters, 0/27063 reco points
  Event 113: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 113: selected reco clusters=0, true clusters=9 (neutrino=1), 1-to-1 neutrino pairs=0, neutrino interactions in mc=3
  Event 114: beam-window cut kept 1/15 reco clusters, 1976/30596 reco points

Found 1 matched pairs of true and re

...

  Event 67: beam-window cut kept 1/13 reco clusters, 1864/22983 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 67: selected reco clusters=1, true clusters=10 (neutrino=1), 1-to-1 neutrino pairs=1, neutrino interactions in mc=1
  Event 68: beam-window cut kept 0/18 reco clusters, 0/33795 reco points
  Event 68: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 68: selected reco clusters=0, true clusters=10 (neutrino=1), 1-to-1 neutrino pairs=0, neutrino interactions in mc=2
  Event 69: beam-window cut kept 0/26 reco clusters, 0/39852 reco points
  Event 69: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 69: selected reco clusters=0, true clusters=15 (neutrino=0), 1-to-1 neutrino pairs=0, neutrino interactions in mc=3

FILE 10/12: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(completeness_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 83: selected reco clusters=1, true clusters=8 (neutrino=0), 1-to-1 neutrino pairs=0, neutrino interactions in mc=1
  Event 84: beam-window cut kept 0/12 reco clusters, 0/36107 reco points
  Event 84: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 84: selected reco clusters=0, true clusters=9 (neutrino=0), 1-to-1 neutrino pairs=0, neutrino interactions in mc=1
  Event 85: beam-window cut kept 3/19 reco clusters, 5030/27999 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 85: selected reco clusters=3, true clusters=9 (neutrino=1), 1-to-1 neutrino pairs=1, neutrino interactions in mc=2
  Event 86: beam-window cut kept 0/22 reco clusters, 0/24661 reco points
  Event 86: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 86: selected reco clusters=0, true clusters=11 (neutrino=0), 1-to-1 neutrino pairs=0, neutrino interactions in mc

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(completeness_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 93: beam-window cut kept 0/17 reco clusters, 0/51567 reco points
  Event 93: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 93: selected reco clusters=0, true clusters=11 (neutrino=0), 1-to-1 neutrino pairs=0, neutrino interactions in mc=2
  Event 94: beam-window cut kept 1/9 reco clusters, 544/25110 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 94: selected reco clusters=1, true clusters=8 (neutrino=1), 1-to-1 neutrino pairs=1, neutrino interactions in mc=1
  Event 95: beam-window cut kept 1/12 reco clusters, 1560/9127 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 95: selected reco clusters=1, true clusters=7 (neutrino=1), 1-to-1 neutrino pairs=1, neutrino interactions in mc=1
  Event 96: beam-window cut kept 1/22 reco clusters, 1641/42054 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 96: selected reco clusters=1, true 